#### Load Dataset

In [1]:
import pandas as pd
import ast

data_path = "../data/usc-x-24-us-election/part_1/"

files = [
    "may_july_chunk_1.csv.gz",
    "may_july_chunk_2.csv.gz",
    "may_july_chunk_3.csv.gz",
    "may_july_chunk_4.csv.gz",
    "may_july_chunk_5.csv.gz"
]

dfs = []
for f in files:
    temp = pd.read_csv(data_path + f, compression='gzip')
    dfs.append(temp)

df = pd.concat(dfs, ignore_index=True)
df.shape

(250000, 32)

In [2]:
df_sample = df[['id', 'text']].dropna().sample(30, random_state=42)
df_sample = df_sample.reset_index(drop=True)
df_sample.shape

(30, 2)

#### Define Issue Descriptions

In [64]:
gallup_issues = {
    "The economy": "inflation, jobs, wages, gas prices, housing costs, cost of living, unemployment, economic growth, recession",
    "Democracy in the U.S.": "elections, voting rights, democracy, election integrity, January 6, Capitol attack, voter fraud, MAGA, democrats, republicans",
    "Terrorism and national security": "terrorism, national security, homeland security, counterterrorism, border security, cyber warfare",
    "Types of Supreme Court justices candidates would pick": "Supreme Court, judicial appointments, Roe v Wade, conservative justices, liberal justices, constitutional law",
    "Immigration": "border, migrants, immigration policy, refugee, alien, visa, H-1B asylum, illegal immigration, deportations, border wall",
    "Education": "schools, colleges, education policy, student loans, parental rights, curriculum, school choice",
    "Healthcare": "health insurance, hospitals, medicine, healthcare costs, Medicare, Obamacare, Medicaid, Affordable Care Act",
    "Gun policy": "guns, gun laws, shootings, second amendment, background checks, assault weapons ban",
    "Abortion": "abortion, reproductive rights, pro-life, pro-choice, IVF, state bans, fetus",
    "Taxes": "taxes, IRS, tax policy, corporate tax, income tax, tax cuts",
    "Crime": "crime, police, violence, law enforcement, public safety, urban crime, fentanyl",
    "Distribution of income and wealth in the U.S.": "wealth gap, income inequality, middle class, billionaire tax, poverty, economic fairness, tax the rich",
    "The federal budget deficit": "national debt, government spending, federal deficit, fiscal policy, debt ceiling",
    "Foreign affairs": "foreign policy, wars, international relations, alliances, NATO, global leadership, diplomacy, dialogue",
    "Situation in Middle East between Israelis and Palestinians": "Israel, Gaza, Palestine, Hamas, Middle East, ceasefire, two-state solution, humanitarian aid, Netanyahu, West Bank, keffiyeh, intifada, islamophobic",
    "Energy policy": "gasoline, drilling, oil, renewable energy, fossil fuels, fracking, energy independence",
    "Relations with Russia": "Russia, Putin, Ukraine war, sanctions, Cold War, Eastern Europe, European Union, EU",
    "Race relations": "racism, racial inequality, discrimination, civil rights, diversity, equity, inclusion, DEI",
    "Relations with China": "China, trade war, Taiwan, competition, TikTok ban, manufacturing, tariffs",
    "Trade with other nations": "trade, tariffs, imports, exports, globalization, outsourcing, protectionism",
    "Climate change": "climate change, environment, energy, emissions, global warming, green new deal",
    "Transgender rights": "transgender, transwoman, cis, LGBTQ, gender identity, gender-affirming care, title IX, bathroom bills, homophobic",
    "Other": "miscellaneous political topic"
}

#### Embed Tweets and Issues

In [4]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

#### Create Issue Embeddings

In [65]:
issue_names = list(gallup_issues.keys())
issue_texts = list(gallup_issues.values())

issue_embeddings = model.encode(issue_texts)

#### Classify Tweets by Similarity

In [56]:
def classify_tweet_semantic(tweet):
    tweet_embedding = model.encode([tweet])
    similarities = cosine_similarity(tweet_embedding, issue_embeddings)[0]
    best_idx = np.argmax(similarities)
    return issue_names[best_idx], similarities[best_idx]

#### Clean Tweets

In [70]:
import re

def clean_tweet(text):
    if text is None:
        return ""
    
    text = re.sub(r'http\S+', '', text)     # remove URLs
    text = re.sub(r'@\w+', '', text)        # remove mentions
    text = re.sub(r'#', '', text)           # remove hashtag symbol
    text = re.sub(r'\n', ' ', text)         # remove newlines
    text = re.sub(r'\s+', ' ', text)        # remove extra spaces
    #text = re.sub(r'[^A-Za-z0-9 ]+', ' ', text) # remove punctuation, symbols, emojis
    text = text.lower().strip()
    
    return text

#### Run on 30 tweets pilot dataframe

In [71]:
df_sample['clean_text'] = df_sample['text'].apply(clean_tweet)

results = df_sample['clean_text'].apply(classify_tweet_semantic)
df_sample[['issue_semantic', 'similarity']] = pd.DataFrame(results.tolist(), index=df_sample.index)
avg_similarity = df_sample['similarity'].mean()
print("Average similarity:", df_sample['similarity'].mean())
print("Max similarity:", df_sample['similarity'].max())
print("Min similarity:", df_sample['similarity'].min())

Average similarity: 0.26142293
Max similarity: 0.48210177
Min similarity: 0.098584965


In [72]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
df_sample.head(5)

,id,text,issue_semantic,clean_text,similarity
0,1801009467170943373,"@kangaroos991 Everything else is in the past. So be it, what you choose to think. But how has Biden saved anything? The most common items are now 5x what they were 4 years ago. He’s just giving trillions away to other nations. Didn’t help Maui at all. Killed Americans, and opened the border???",The economy,"everything else is in the past. so be it, what you choose to think. but how has biden saved anything? the most common items are now 5x what they were 4 years ago. he’s just giving trillions away to other nations. didn’t help maui at all. killed americans, and opened the border???",0.320805
1,1800990362770509922,"Man who manipulates rates, thus inflation to help Biden get re-elected wonders why so many are mad?",The economy,"man who manipulates rates, thus inflation to help biden get re-elected wonders why so many are mad?",0.380497
2,1801038374267969811,@Coloradosearch1 @JoJoFromJerz But unlike Donald Trump he is not a CONVICTED FELON or rapist,Gun policy,but unlike donald trump he is not a convicted felon or rapist,0.312078
3,1800948689441259936,@TrumpSolMAGA @WhaleInsider preparing for a massive bull 🔥$maga,Other,preparing for a massive bull 🔥$maga,0.214100
4,1800910374780481866,"Biden reportedly puts the blames for Hunter's conviction on his re-election bid. Saying, ""He would have gotten the plea deal.""\nNow, that claim didn't work for Trump, when Trump's case was really happening.\nIf Biden thinks this will fly W/ voters, he's got another thing coming.",Gun policy,"biden reportedly puts the blames for hunter's conviction on his re-election bid. saying, ""he would have gotten the plea deal."" now, that claim didn't work for trump, when trump's case was really happening. if biden thinks this will fly w/ voters, he's got another thing coming.",0.209390


In [ ]:
try:
    df_sample.to_csv("../data/sample_issue_classification.csv", index=False)
    print("Saved dataset: ../data/sample_issue_classification.csv")
except Exception as e:
    print("Error while saving:", e)

#### Run on entire df

In [73]:
df['clean_text'] = df['text'].apply(clean_tweet)

all_results = df['clean_text'].apply(classify_tweet_semantic)
df[['issue_semantic', 'similarity']] = pd.DataFrame(all_results.tolist(), index=df.index)
print("Average similarity:", df['similarity'].mean())
print("Max similarity:", df['similarity'].max())
print("Min similarity:", df['similarity'].min())

Average similarity: 0.24372949
Max similarity: 0.7278557
Min similarity: -0.048169218


In [74]:
df.head()

,Unnamed: 0,id,text,url,epoch,media,retweetedTweet,retweetedTweetID,retweetedUserID,id_str,lang,rawContent,replyCount,retweetCount,likeCount,quoteCount,conversationId,conversationIdStr,hashtags,mentionedUsers,links,viewCount,quotedTweet,in_reply_to_screen_name,in_reply_to_status_id_str,in_reply_to_user_id_str,location,cash_app_handle,user,date,_type,type,clean_text,issue_semantic,similarity
0,0,1801041792923578484,"@lukepbeasley I cant imagine anyone actually feels this way. As much as you hate Donald trump, you don’t believe this. Brainwashed, complicit (black mail or money), or just have no clue whats going in and just want to fit in somewhere. There are no other options",https://twitter.com/orgneyezedchaos/status/1801041792923578484,1.718237e+09,[],False,NaN,NaN,1801041792923578484,en,"@lukepbeasley I cant imagine anyone actually feels this way. As much as you hate Donald trump, you don’t believe this. Brainwashed, complicit (black mail or money), or just have no clue whats going in and just want to fit in somewhere. There are no other options",0,0.0,1.0,0.0,1.800681e+18,1.800681e+18,[],"[{'id_str': '1483596463221264387', 'name': 'Luke Beasley', 'screen_name': 'lukepbeasley', 'indices': [0, 13]}]",[],"{'count': '10', 'state': 'EnabledWithCount'}",False,lukepbeasley,1.800681e+18,1.483596e+18,NaN,NaN,"{'id': 942869257108455424, 'id_str': '942869257108455424', 'url': 'https://twitter.com/B. G. Dalena', 'username': 'B. G. Dalena', 'rawDescription': 'Therapist. Philosophy, Epistemology, Politics, Neuroscience, Golf, Art, Sports, and a relentless pursuit of the truth.', 'created': datetime.datetime(2017, 12, 18, 21, 28, 43, tzinfo=datetime.timezone.utc), 'followersCount': 152, 'friendsCount': 246, 'statusesCount': 390, 'favouritesCount': 19, 'listedCount': 0, 'mediaCount': 1, 'location': 'San Diego, CA', 'profileImageUrl': 'https://pbs.twimg.com/profile_images/1613647617594130432/h3jM9GzR_normal.jpg', 'profileBannerUrl': 'PW', 'protected': 'PW', 'verified': False, 'blue': False, 'blueType': None, 'descriptionLinks': ['PW'], '_type': 'PW'}",2024-06-12,NaN,tweet-,"i cant imagine anyone actually feels this way. as much as you hate donald trump, you don’t believe this. brainwashed, complicit (black mail or money), or just have no clue whats going in and just want to fit in somewhere. there are no other options",Immigration,0.160803
1,1,1801041792630227173,"Voters can also sway me away from voting for someone. @Potomacbeat , the way I see people on the right supporting gun control just bc it's a political Oppenent also makes me just consider voting in local and state where I'll get my principled conservative and policies.",https://twitter.com/Brandon62294232/status/1801041792630227173,1.718237e+09,[],False,NaN,NaN,1801041792630227173,en,"Voters can also sway me away from voting for someone. @Potomacbeat , the way I see people on the right supporting gun control just bc it's a political Oppenent also makes me just consider voting in local and state where I'll get my principled conservative and policies.",1,0.0,0.0,0.0,1.801042e+18,1.801042e+18,[],"[{'id_str': '423966379', 'name': 'CONSERVATIVE THOUGHT - 𝙏𝙃𝙀 𝘿𝘾 𝙋𝙍𝙊𝙋𝙃𝙀𝙏', 'screen_name': 'Potomacbeat', 'indices': [55, 67]}]",[],"{'count': '40', 'state': 'EnabledWithCount'}",False,NaN,NaN,NaN,NaN,NaN,"{'id': 1461100431329796100, 'id_str': '1461100431329796100', 'url': 'https://twitter.com/Brandon Holloway', 'username': 'Brandon Holloway', 'rawDescription': 'conservative. principled. ideology over any man or Politician. 2A advocate. I support policy, and \n ideology', 'created': datetime.datetime(2021, 11, 17, 22, 34, 42, tzinfo=datetime.timezone.utc), 'followersCount': 518, 'friendsCount': 719, 'statusesCount': 19678, 'favouritesCount': 3413, 'listedCount': 0, 'mediaCount': 1002, 'location': '', 'profileImageUrl': 'https://pbs.twimg.com/profile_images/1684111500036767744/kqiShB8P_normal.jpg', 'profileBannerUrl': 'PW', 'protected': 'PW', 'verified': False, 'blue': False, 'blueType': None, '

In [75]:
try:
    df.to_csv("../data/issue_classification.csv", index=False)
    print("Saved dataset: ../data/issue_classification.csv")
except Exception as e:
    print("Error while saving:", e)

Saved dataset: ../data/issue_classification.csv


#### Aggregate Tweets by Issue

In [76]:
tweets_per_issue = df.groupby('issue_semantic').size().sort_values(ascending=False)
print(tweets_per_issue)

users_per_issue = df.groupby('issue_semantic')['user'].nunique().sort_values(ascending=False)
print(users_per_issue)

similarity_per_issue = df.groupby('issue_semantic')['similarity'].mean().sort_values(ascending=False)
print(similarity_per_issue)

issue_semantic
Democracy in the U.S.                                         58761
Other                                                         31312
Types of Supreme Court justices candidates would pick         23277
Gun policy                                                    16208
Foreign affairs                                               15070
Situation in Middle East between Israelis and Palestinians    14780
Immigration                                                   11738
Relations with Russia                                         10359
The economy                                                    9089
Climate change                                                 7114
Distribution of income and wealth in the U.S.                  7079
Transgender rights                                             6938
Crime                                                          6347
Race relations                                                 4838
Relations with China             

In [77]:
issue_summary = pd.DataFrame({
    'tweet_count': df.groupby('issue_semantic').size(),
    'unique_users': df.groupby('issue_semantic')['user'].nunique(),
    'avg_similarity': df.groupby('issue_semantic')['similarity'].mean()
})

issue_summary = issue_summary.sort_values(by='tweet_count', ascending=False)

print(issue_summary)

                                                            tweet_count  \
issue_semantic                                                            
Democracy in the U.S.                                             58761   
Other                                                             31312   
Types of Supreme Court justices candidates would pick             23277   
Gun policy                                                        16208   
Foreign affairs                                                   15070   
Situation in Middle East between Israelis and Palestinians        14780   
Immigration                                                       11738   
Relations with Russia                                             10359   
The economy                                                        9089   
Climate change                                                     7114   
Distribution of income and wealth in the U.S.                      7079   
Transgender rights       

In [78]:
try:
    issue_summary.to_csv("../data/issue_summary.csv", index=False)
    print("Saved dataset: ../data/issue_summary.csv")
except Exception as e:
    print("Error while saving:", e)

Saved dataset: ../data/issue_summary.csv


In [81]:
import plotly.express as px

issue_counts = df['issue_semantic'].value_counts().reset_index()
issue_counts.columns = ['Issue', 'Tweet Count']

fig = px.bar(
    issue_counts.sort_values('Tweet Count'),
    x='Tweet Count',
    y='Issue',
    orientation='h',
    text='Tweet Count',
    title='Tweets per Issue (Semantic Classification)'
)

fig.update_layout(
    yaxis=dict(categoryorder='total ascending'),
    height=600
)

fig.show()

In [82]:
top_issues = issue_counts.sort_values('Tweet Count', ascending=False).head(10)

fig = px.bar(
    top_issues.sort_values('Tweet Count'),
    x='Tweet Count',
    y='Issue',
    orientation='h',
    text='Tweet Count',
    title='Top 10 Issues Discussed on Twitter'
)

fig.show()